# integración ontológica carga académica

In [1]:
# Importación de librerías
import os
%pip install owlready2
import owlready2 as owl
from crewai import Agent, Crew, Process, Task, LLM
from crewai.tools import tool

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# configuración del modelo
llm = LLM(
    model = "ollama/llama3.2",
    base_url="http://localhost:11434"
)

In [3]:
@tool("consulta de ontología académica")
def consultar_onto_academica(tipo_consulta: str, entidad_id: str = "") -> str:
    """ 
    Consulta sobre la base de conocimiento modelada en la ontología académica de una universidad (ontologia-ia-ejemplo-1.owx).
    Donde los parámetros son:
    - tipo_consulta: Es una cadena que indica el motivo de la consulta (ej. 'buscar_alumno', 'buscar_materias_aprobadas', 'buscar_materias_disponibles')
    - entidad_id: [opcional] Es el identificador o matrícula de un individuo en particular, por ejemplo: alumno_001, A001, etc.
    """
    ruta_onto = "P2_ontologia_carga_academica (1).owx"
    if not os.path.exists(ruta_onto):
        return f"Ocurrió un error al buscar la ontología en la ruta {ruta_onto}, por favor verifica que exista."
    
    try:
        onto = owl.get_ontology(ruta_onto).load()
        
        match tipo_consulta:
            case "buscar_alumno":
                # Busca por coincidencia en la matrícula o en el IRI del individuo
                alumno = onto.search_one(matricula=entidad_id) or onto.search_one(iri=f"*{entidad_id}*")
                if alumno:
                    nombre = alumno.nombrePersona[0] if hasattr(alumno, "nombrePersona") and alumno.nombrePersona else "sin definir"
                    matricula = alumno.matricula[0] if hasattr(alumno, "matricula") and alumno.matricula else "sin definir"
                    return f"El alumno con ID {alumno.name}, con nombre {nombre} y matrícula {matricula} fue encontrado con éxito."
                else:
                    return f"El alumno con identificador '{entidad_id}' no fue encontrado."
                
            case "buscar_materias_aprobadas":
                alumno = onto.search_one(matricula=entidad_id) or onto.search_one(iri=f"*{entidad_id}*")
                if alumno and hasattr(alumno, "haAprobado"):
                    materias = [mat.name for mat in alumno.haAprobado]
                    if materias:
                        return f"El alumno {entidad_id} ha aprobado las siguientes materias: {', '.join(materias)}."
                    else:
                        return f"El alumno {entidad_id} no registra materias aprobadas."
                else:
                    return f"No se encontró al alumno {entidad_id} o no cuenta con registros de aprobación."
                
            case "buscar_materias_disponibles":
                alumno = onto.search_one(matricula=entidad_id) or onto.search_one(iri=f"*{entidad_id}*")
                if alumno and hasattr(alumno, "puedeCursar"):
                    materias = [mat.name for mat in alumno.puedeCursar]
                    if materias:
                        return f"El alumno {entidad_id} cumple con los requisitos para cursar: {', '.join(materias)}."
                    else:
                        return f"El alumno {entidad_id} no tiene materias disponibles para cursar actualmente."
                else:
                    return f"No se encontró al alumno {entidad_id} en la ontología."
                    
            case _:
                return "No se reconoce el tipo de búsqueda solicitado."
            
    except Exception as e:
        return f"Ocurrió un error al procesar la ontología: {str(e)}"

### Definición de los agentes

In [4]:
# Agente 1:Analista de historial y requisitos
agente_historial = Agent(
    role="Analista de historial académico y requisitos",
    goal="...",
    backstory="...",
    tools=[consultar_onto_academica],  # <-- Asegúrate de que la tenga asignada
    llm=llm,
    verbose=True
)

In [5]:
# Agente 2: Analista de carga académica
agente_carga = Agent(
    role="Analista de carga académica",
    goal="Evaluar las combinaciones posibles de materias y determinar cuáles representan una carga equilibrada para el estudiante.",
    backstory=(
        "Este agente analiza la cantidad de materias y créditos que forman parte de cada propuesta. "
        "Busca evitar cargas excesivas y considera las restricciones académicas del estudiante. "
        "Su función es encontrar un equilibrio entre el avance académico y la cantidad de trabajo que representa la carga."
    ),
    tools=[consultar_onto_academica],  # Puede consultar la ontología para verificar créditos totales del plan y límites de carga
    llm=llm,
    verbose=True
)

In [6]:
#Agente 3: Analista de horarios y preferencias
agente_horarios = Agent(
    role="Analista de horarios y preferencias del estudiante",
    goal="Evaluar la compatibilidad de horarios y adaptar las propuestas a las preferencias expresadas por el estudiante.",
    backstory=(
        "Este agente analiza los horarios, modalidades y preferencias del alumno. "
        "Puede interpretar preferencias cualitativas como querer una carga ligera, clases por la tarde o clases en línea. "
        "Esta función mantiene la idea original de utilizar un modelo de lenguaje para interpretar las preferencias expresadas mediante lenguaje natural."
    ),
    tools=[consultar_onto_academica],  # Herramienta para consultar grupos, turnos, horarios y conflictos
    llm=llm,
    verbose=True
)

In [7]:
# Agente 4: manager/coordinador
agente_manager = Agent(
    role="Coordinador y recomendador académico",
    goal="Coordinar a los agentes especializados, comparar sus resultados y seleccionar la recomendación de carga académica que mejor cumpla las restricciones y preferencias del estudiante.",
    backstory=(
        "Este agente representa la evolución del agente asesor original hacia una arquitectura multiagente. "
        "Es un coordinador académico virtual encargado de integrar información proveniente de diferentes especialistas. "
        "Recibe los resultados de los agentes de historial, carga académica y horarios/preferencias, "
        "compara las propuestas y genera la recomendación final mediante la capa de coordinación."
    ),
    llm=llm,
    verbose=True
)

### Flujo de trabajo

In [8]:
tarea_historial = Task(
    description=(
        "Consulta dentro de la ontología académica para analizar el historial del estudiante (ej. alumno_001). "
        "Debes realizar lo siguiente:\n"
        "- Consultar las materias aprobadas y reprobadas del alumno.\n"
        "- Verificar los prerrequisitos y reglas de seriación institucionales.\n"
        "- Determinar qué materias válidas puede cursar el estudiante en el periodo actual."
    ),
    expected_output=(
        "Un reporte estructurado con el kárdex del alumno, la lista de materias válidas que "
        "cumplen con la seriación y los prerrequisitos institucionales."
    ),
    agent=agente_historial
)

In [9]:
tarea_carga = Task(
    description=(
        "Evaluar las combinaciones posibles de materias para el estudiante (ej. alumno_001) "
        "utilizando las reglas de la ontología. Debes:\n"
        "- Analizar el total de créditos acumulados y la meta del plan de estudios.\n"
        "- Evaluar el número de materias y los límites de créditos permitidos por periodo (carga mínima y máxima).\n"
        "- Asegurar que la combinación propuesta represente una carga equilibrada."
    ),
    expected_output=(
        "Un reporte de carga académica que desglose el número de créditos, la cantidad de materias "
        "proyectadas y la validación de que se ajusta a los límites institucionales."
    ),
    agent=agente_carga
)

In [10]:
tarea_horarios = Task(
    description=(
        "Analizar la disponibilidad de horarios y las preferencias expresadas por el estudiante (ej. alumno_001). "
        "Debes:\n"
        "- Consultar la oferta de grupos, días, horas y profesores disponibles en la ontología.\n"
        "- Detectar posibles conflictos o cruces de horarios entre los grupos candidatos.\n"
        "- Interpretar las preferencias del alumno (como clases por la tarde, modalidad o carga ligera) "
        "para alinear la propuesta a sus necesidades."
    ),
    expected_output=(
        "Un reporte de compatibilidad de horarios que confirme la ausencia de cruces y el nivel "
        "de alineación con las preferencias cualitativas del estudiante."
    ),
    agent=agente_horarios
)

In [11]:
tarea_coordinacion = Task(
    description=(
        "Actuar como coordinador bajo el protocolo Contract Net. Debes:\n"
        "- Recopilar y comparar las propuestas de los agentes especializados.\n"
        "- Evaluar las alternativas y seleccionar la carga académica óptima.\n"
        "- IMPORTANTE: No imprimas llamadas a herramientas ni objetos JSON técnicos en el resultado final. "
        "Presenta la recomendación en un reporte redactado en lenguaje natural y claro para el estudiante."
    ),
    expected_output=(
        "Un informe final redactado que detalle la lista de materias recomendadas, "
        "el total de créditos y una justificación clara basada en la evaluación de los agentes."
    ),
    agent=agente_manager,
    context=[tarea_historial, tarea_carga, tarea_horarios]
)

### Creación del equipo de trabajo

In [12]:
equipo_academico = Crew(
    agents=[
        agente_historial,  # Agente 1: Analista de Historial y Requisitos
        agente_carga,      # Agente 2: Analista de Carga Académica
        agente_horarios    # Agente 3: Analista de Horarios y Preferencias
    ],
    tasks=[
        tarea_historial,   # Tarea para el análisis de materias válidas e historial
        tarea_carga,       # Tarea para la evaluación de créditos y carga de trabajo
        tarea_horarios,    # Tarea para la detección de conflictos y preferencias
        tarea_coordinacion # Tarea de integración y recomendación final
    ],
    manager_agent=agente_manager,  # Agente 4: Manager / Coordinador principal
    process=Process.hierarchical,  # Usamos proceso jerárquico según el diseño del proyecto
    verbose=True
)

In [13]:
async def ejecutar_tareas():
    print("Iniciando la ejecución del Sistema Multiagente...")
    resultado = await equipo_academico.kickoff_async()
    print("\n RESULTADO FINAL:")
    print(resultado)

In [14]:
await ejecutar_tareas()

Iniciando la ejecución del Sistema Multiagente...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 642c8446-2712-4ce6-a1ff-00f0edce8cac                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Consulta dentro de la ontología académica para analizar el historial del estudiante (ej. alumno_001).    │
│  Debes realizar lo siguiente:                                                                                   │
│  - Consultar las materias aprobadas y reprobadas del alumno.                                                    │
│  - Verificar los prerrequisitos y reglas de seriación institucionales.                                          │
│  - Determinar qué materias válidas puede cursar el estudiante en el periodo actual.                             │
│  ID: 9eed9224-fb64-405e-98fc-afdd36e4d207                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Coordinador y recomendador académico                                                                    │
│                                                                                                                 │
│  Task: Consulta dentro de la ontología académica para analizar el historial del estudiante (ej. alumno_001).    │
│  Debes realizar lo siguiente:                                                                                   │
│  - Consultar las materias aprobadas y reprobadas del alumno.                                                    │
│  - Verificar los prerrequisitos y reglas de seriación institucionales.                                          │
│  - Determinar qué materias válidas puede cursar el estudiante en el periodo actual.                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Using an approximate input token count for model 'llama3.2': assuming one token per four characters because a native token count is unavailable.
The context window of model 'llama3.2' is unknown, so CrewAI assumes 8,192 tokens (6,963 usable). If the model's window is different, pass it as LLM(model=..., context_window_size=<tokens>).
ERROR:root:OpenAI API call failed: Error code: 500 - {'error': {'message': 'llama runner process has terminated: ROCm error: invalid device function\n  current device: 0, in function ggml_cuda_compute_forward at /builddir/build/BUILD/ollama-0.12.11-build/ollama-0.12.11/ml/backend/ggml/ggml/src/ggml-cuda/ggml-cuda.cu:2672\n  err\n/builddir/build/BUILD/ollama-0.12.11-build/ollama-0.12.11/ml/backend/ggml/ggml/src/ggml-cuda/ggml-cuda.cu:88: ROCm error', 'type': 'api_error', 'param': None, 'code': None}}
ERROR:root:OpenAI API call failed: Error code: 500 - {'error': {'message': 'llama runner process has terminated: ROCm error: invalid device function\n  current

[CrewAIEventsBus] Warning: Event pairing mismatch. 'llm_call_failed' closed 'flow_started' (expected 
'llm_call_started')

... (20 outputs left out)

An unknown error occurred. Please check the details below.
Error details: Error code: 500 - {'error': {'message': 'llama runner process has terminated: ROCm error: invalid device function\n  current device: 0, in function ggml_cuda_compute_forward at /builddir/build/BUILD/ollama-0.12.11-build/ollama-0.12.11/ml/backend/ggml/ggml/src/ggml-cuda/ggml-cuda.cu:2672\n  err\n/builddir/build/BUILD/ollama-0.12.11-build/ollama-0.12.11/ml/backend/ggml/ggml/src/ggml-cuda/ggml-cuda.cu:88: ROCm error', 'type': 'api_error', 'param': None, 'code': None}}


╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Consulta dentro de la ontología académica para analizar el historial del estudiante (ej. alumno_001).    │
│  Debes realizar lo siguiente:                                                                                   │
│  - Consultar las materias aprobadas y reprobadas del alumno.                                                    │
│  - Verificar los prerrequisitos y reglas de seriación institucionales.                                          │
│  - Determinar qué materias válidas puede cursar el estudiante en el periodo actual.                             │
│  Agent: Coordinador y recomendador académico                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 642c8446-2712-4ce6-a1ff-00f0edce8cac                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

InternalServerError: Error code: 500 - {'error': {'message': 'llama runner process has terminated: ROCm error: invalid device function\n  current device: 0, in function ggml_cuda_compute_forward at /builddir/build/BUILD/ollama-0.12.11-build/ollama-0.12.11/ml/backend/ggml/ggml/src/ggml-cuda/ggml-cuda.cu:2672\n  err\n/builddir/build/BUILD/ollama-0.12.11-build/ollama-0.12.11/ml/backend/ggml/ggml/src/ggml-cuda/ggml-cuda.cu:88: ROCm error', 'type': 'api_error', 'param': None, 'code': None}}